# Colab Multitask Dense Baseline

Notebook này train baseline multitask **không dùng MoE** trên hai task `DocVQA` + `ChartQA` với một backbone chung `Pix2Struct`.

Mục tiêu của notebook:
- tạo một dense baseline chung để sau này so với MoE
- giữ pipeline Colab-first, không phụ thuộc local
- trộn hai task trong cùng một run


In [ ]:
!python -V
!nvidia-smi


In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio transformers datasets pillow sentencepiece accelerate


In [ ]:
import os
from pathlib import Path

import torch
from datasets import concatenate_datasets, load_dataset
from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor, Trainer, TrainingArguments

WORKDIR = Path('/content/moe_multitask_dense')
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print('Working directory:', WORKDIR)


In [ ]:
# Change only these config values.
MODEL_NAME = 'google/pix2struct-base'
DOCVQA_SAMPLE_SIZE = 1200
CHARTQA_SAMPLE_SIZE = 1200
VAL_SIZE = 0.1
SEED = 42

BATCH_SIZE = 2
LEARNING_RATE = 5e-5
NUM_EPOCHS = 2
WEIGHT_DECAY = 0.01

MAX_PATCHES = 1024
MAX_ANSWER_LENGTH = 48

HF_TOKEN = ''  # optional
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

print('MODEL_NAME =', MODEL_NAME)
print('DOCVQA_SAMPLE_SIZE =', DOCVQA_SAMPLE_SIZE)
print('CHARTQA_SAMPLE_SIZE =', CHARTQA_SAMPLE_SIZE)


In [ ]:
def normalize_text(text):
    if text is None:
        return ''
    return ' '.join(str(text).strip().split())


def first_answer(value):
    if isinstance(value, list):
        return value[0] if value else ''
    return value if value is not None else ''


def build_labels(token_ids, pad_token_id):
    return [[-100 if token_id == pad_token_id else token_id for token_id in row] for row in token_ids]


def sample_dataset(dataset, sample_size, seed):
    sample_size = min(sample_size, len(dataset))
    return dataset.shuffle(seed=seed).select(range(sample_size))


## Download and normalize DocVQA + ChartQA

In [ ]:
doc_ds = load_dataset('lmms-lab/DocVQA', 'DocVQA')
doc_split = doc_ds['train'] if 'train' in doc_ds else doc_ds[next(iter(doc_ds.keys()))]
doc_split = sample_dataset(doc_split, DOCVQA_SAMPLE_SIZE, SEED)
doc_dataset = doc_split.map(
    lambda ex: {
        'task': 'docvqa',
        'image': ex['image'],
        'question': normalize_text(ex['question']),
        'answer': normalize_text(first_answer(ex['answers'])),
    },
    remove_columns=doc_split.column_names,
)

chart_ds = load_dataset('HuggingFaceM4/ChartQA')
chart_split = chart_ds['train'] if 'train' in chart_ds else chart_ds[next(iter(chart_ds.keys()))]
chart_split = sample_dataset(chart_split, CHARTQA_SAMPLE_SIZE, SEED)
chart_dataset = chart_split.map(
    lambda ex: {
        'task': 'chartqa',
        'image': ex['image'],
        'question': normalize_text(ex['query']),
        'answer': normalize_text(first_answer(ex['label'])),
    },
    remove_columns=chart_split.column_names,
)

task_dataset = concatenate_datasets([doc_dataset, chart_dataset]).shuffle(seed=SEED)
print(task_dataset)
task_dataset.select(range(5)).to_pandas()


In [ ]:
dataset = task_dataset.train_test_split(test_size=VAL_SIZE, seed=SEED)
print(dataset)


## Preprocess for a shared Pix2Struct model

In [ ]:
processor = Pix2StructProcessor.from_pretrained(MODEL_NAME)
model = Pix2StructForConditionalGeneration.from_pretrained(MODEL_NAME)

def preprocess_batch(batch):
    prompted_questions = [
        f'[{task}] question: {normalize_text(question)}'
        for task, question in zip(batch['task'], batch['question'])
    ]
    answers = [normalize_text(answer) for answer in batch['answer']]

    model_inputs = processor(
        images=batch['image'],
        text=prompted_questions,
        truncation=True,
        max_patches=MAX_PATCHES,
    )

    answer_tokens = processor.tokenizer(
        answers,
        padding='max_length',
        truncation=True,
        max_length=MAX_ANSWER_LENGTH,
    )

    model_inputs['labels'] = build_labels(answer_tokens['input_ids'], processor.tokenizer.pad_token_id)
    return model_inputs

MAP_BATCH_SIZE = 8

train_dataset = dataset['train'].map(
    preprocess_batch,
    batched=True,
    batch_size=MAP_BATCH_SIZE,
    writer_batch_size=MAP_BATCH_SIZE,
    remove_columns=dataset['train'].column_names,
)

val_dataset = dataset['test'].map(
    preprocess_batch,
    batched=True,
    batch_size=MAP_BATCH_SIZE,
    writer_batch_size=MAP_BATCH_SIZE,
    remove_columns=dataset['test'].column_names,
)

train_dataset.set_format(type='torch')
val_dataset.set_format(type='torch')

print(train_dataset)
print(val_dataset)


## Train dense multitask baseline

In [ ]:
OUTPUT_DIR = WORKDIR / 'pix2struct_multitask_dense_outputs'

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()
metrics = trainer.evaluate()
metrics


In [ ]:
trainer.save_model(str(OUTPUT_DIR / 'final_model'))
processor.save_pretrained(str(OUTPUT_DIR / 'final_model'))
print('Saved model to', OUTPUT_DIR / 'final_model')
